# Classifying with Scikit-learn

In regression, a model predicts a *number*. **Classification** predicts a *category* instead: pass or fail, cat or dog, spam or not spam.

| Question | Kind | Answer looks like |
|---|---|---|
| What score will this student get? | Regression | 84.5 |
| Will this student pass? | Classification | Pass / Fail |
| How hot will it be tomorrow? | Regression | 22 degrees |
| Will it rain tomorrow? | Classification | Rain / No rain |

The three steps are exactly the same as before: **prepare**, **fit**, **predict**.

# The Dataset

Question: *can a machine predict whether a student passes, from how much they studied and how much they slept?*

This time there are **two features** (hours studied, hours slept), and the label is Pass (`1`) or Fail (`0`).

In [ ]:
import pandas as pd

data = {
  'hours_studied': [2, 3, 1, 7, 5, 6, 8, 9, 4, 7, 3, 6, 8, 2, 5, 4, 9, 1, 7, 6],
  'hours_slept':   [4, 5, 8, 4, 7, 8, 6, 7, 6, 7, 3, 5, 8, 7, 9, 4, 5, 5, 9, 6],
  'passed':        [0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1]
}

df = pd.DataFrame(data)
df['result'] = df['passed'].map({1: 'Pass', 0: 'Fail'})
df

# Features and Labels

Split the data the same way as before, but with two feature columns:

- **X**, the *features*: hours studied **and** hours slept
- **y**, the *label*: 1 for Pass, 0 for Fail

With two features the model can weigh study time and sleep together.

In [ ]:
X = df[['hours_studied', 'hours_slept']]
y = df['passed']

print("X (features):")
print(X.head(3))
print()
print("y (labels):", list(y))

# Training a Classifier

We'll use **K-Nearest Neighbors** (KNN), one of the easiest classifiers to picture.

To classify a new student, KNN finds the `k` students in the data who are most similar and takes a majority vote. With `n_neighbors=3` it looks at the 3 closest students: if 2 of them passed, it predicts Pass.

`model.score(X, y)` reports **accuracy**: how often the model gets the answer right.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)

accuracy = model.score(X, y)
print(f"Training accuracy: {accuracy * 100:.0f}%")

# Making Predictions

Set a new student's hours with the sliders and see the model's verdict.

In [ ]:
HOURS_STUDIED = 5 #@param {type:"slider", min:1, max:10, step:1}
HOURS_SLEPT   = 7 #@param {type:"slider", min:1, max:10, step:1}

new_student = pd.DataFrame({'hours_studied': [HOURS_STUDIED], 'hours_slept': [HOURS_SLEPT]})
prediction = model.predict(new_student)[0]

print(f"Studied {HOURS_STUDIED} h, slept {HOURS_SLEPT} h")
print("Prediction:", "Pass" if prediction == 1 else "Fail")

# Seeing the Decision Boundary

With two features, every student is a point on a chart. The shaded regions show what the model would predict *anywhere* on the chart, and the edge between them is the **decision boundary**.

Change `K` and watch the boundary change shape. A small `K` follows individual points closely; a large `K` gives a smoother, more general rule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

K = 3 #@param {type:"slider", min:1, max:15, step:1}

model_k = KNeighborsClassifier(n_neighbors=K)
model_k.fit(X, y)

gx, gy = np.meshgrid(np.linspace(0, 11, 200), np.linspace(0, 11, 200))
zone = model_k.predict(pd.DataFrame({'hours_studied': gx.ravel(), 'hours_slept': gy.ravel()}))
zone = zone.reshape(gx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(gx, gy, zone, alpha=0.15, levels=[-0.5, 0.5, 1.5], colors=['#e57373', '#81c784'])
passed = df[df['passed'] == 1]
failed = df[df['passed'] == 0]
plt.scatter(passed['hours_studied'], passed['hours_slept'], color='#2e7d32', s=80, label='Pass')
plt.scatter(failed['hours_studied'], failed['hours_slept'], color='#c62828', s=80, label='Fail')
plt.title(f'Decision boundary with K = {K}')
plt.xlabel('Hours Studied')
plt.ylabel('Hours Slept')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

print(f"Accuracy with K={K}: {model_k.score(X, y) * 100:.0f}%")

# Check Your Understanding